# 🔤 Syndicate 3.0: Phase 3 Playroom (The Secret Language of Tokens)

LLMs do not see characters, and they do not see words. They see **tokens**—numerical indices pointing to a fixed vocabulary. 

In this playroom, you will master the mechanics of tokenization. We will build a functioning **Byte-Pair Encoding (BPE)** tokenizer from scratch using pure Python primitives. You will see exactly how subword merge mechanics allow a model to represent complex language with a finite vocabulary.

---

## 🔢 3.0: Encodings, Bytes, and String Physics

Before we can train a tokenizer to merge bytes, we must understand how strings exist in computer memory. 
In Python, a string (`str`) is an abstract sequence of Unicode characters. To transmit or store it, it must be encoded into a raw sequence of **bytes** (numbers from `0` to `255`).

### 🧪 Discovery Lab: The String to Byte Translation

#### 1. The Unicode Code Points
Every character has a unique integer ID called a Unicode code point. We can find this integer in Python using `ord()`:
```python
ord('A') # Returns 65
ord('👋') # Returns 128075 (Requires multiple bytes to store!)
```

#### 2. UTF-8 Byte Encoding
UTF-8 is the standard variable-length encoding that translates Unicode code points into bytes. ASCII characters take 1 byte, while emojis can take up to 4 bytes.
```python
"A".encode("utf-8") # Returns b'A' (length 1 byte)
"👋".encode("utf-8") # Returns b'\xf0\x9f\x91\x8b' (length 4 bytes: [240, 159, 145, 139])
```

In [ ]:
# TODO: Write a utility function that takes a standard Python string
# and returns a list of its raw byte integer values (0 to 255) using UTF-8 encoding.
def string_to_bytes(text: str) -> list[int]:
    # Hint: use text.encode('utf-8') to get bytes, and convert each byte in the sequence to an int.
    return []

# Verification:
sample = "Hello 👋"
byte_ints = string_to_bytes(sample)
print(f"Raw byte integers: {byte_ints}")

# 'Hello ' is 6 bytes (ASCII) + '👋' is 4 bytes = 10 bytes total
# once implemented correctly, this assertion passes:
# assert len(byte_ints) == 10
# assert byte_ints[:5] == [72, 101, 108, 108, 111] # 'Hello'
print("String-to-bytes physics lab loaded.")

## 🔢 3.1: Subword Frequencies & BPE Concept

Why not character-level tokenization? Character tokens (e.g. `'a'`, `'b'`) require long sequence lengths, making memory utilization explode. 
Why not word-level tokenization? A dictionary of every possible word in the English language grows to millions of entries, wasting model parameters on vocabulary lookup tables and crashing when it encounters a typo (out-of-vocabulary words).

**Byte-Pair Encoding (BPE)** solves this. It starts with individual characters as a vocabulary and iteratively merges the most frequent adjacent token pairs in a text corpus to build larger tokens (subwords or whole words).

Let's trace this merge process by hand in Python.

In [ ]:
from typing import Dict, Tuple, List

# Sample corpus tokenized into characters (each word ending with a special end-of-word token '</w>')
corpus = [
    ["h", "u", "g", "</w>"],
    ["p", "u", "g", "</w>"],
    ["h", "u", "g", "s", "</w>"],
    ["h", "u", "g", "</w>"],
    ["p", "u", "g", "</w>"]
]

# TODO: Write a function to count frequencies of adjacent pairs of tokens in the corpus.
def get_stats(corpus: List[List[str]]) -> Dict[Tuple[str, str], int]:
    pairs = {}
    # Loop over corpus and count frequency of consecutive items
    return pairs

stats = get_stats(corpus)
print(f"Pair statistics: {stats}")

---
## ⚙️ 3.2: Merging Token Pairs

Once we identify the most frequent pair of tokens, we merge them into a single token (e.g., merging `('h', 'u')` into `'hu'`). We repeat this process until the desired vocabulary size is reached.

### 🧪 Discovery Lab: The Merge Loop

In [ ]:
# TODO: Write a function that takes a pair of tokens (e.g. ('h', 'u')) and merges them in every sequence in the corpus.
def merge_pair(pair: Tuple[str, str], corpus: List[List[str]]) -> List[List[str]]:
    new_corpus = []
    # Iterate through corpus sequences, replacing consecutive tokens in 'pair' with their concatenated form.
    return new_corpus

# Test your merge function
target_pair = ("h", "u")
merged_corpus = merge_pair(target_pair, corpus)
print(f"Merged corpus: {merged_corpus}")
# Expected: The first sequence should be ['hu', 'g', '</w>']

---
## 🛠 3.3: BPE Vocabulary Generation

Now we can tie statistics and merging into a loop to dynamically train a BPE Vocabulary.

In [ ]:
# TODO: Write a toy BPE trainer that creates a dictionary of merges
def train_bpe(corpus: List[List[str]], num_merges: int) -> Tuple[List[str], Dict[Tuple[str, str], str]]:
    vocab = set(char for seq in corpus for char in seq)
    merges = {}
    
    for i in range(num_merges):
        stats = get_stats(corpus)
        if not stats:
            break
        best_pair = max(stats, key=stats.get)
        new_token = "".join(best_pair)
        corpus = merge_pair(best_pair, corpus)
        vocab.add(new_token)
        merges[best_pair] = new_token
        print(f"Merge {i+1}: {best_pair} -> {new_token}")
        
    return list(vocab), merges

final_vocab, final_merges = train_bpe(corpus, 3)
print(f"\nFinal Vocabulary: {final_vocab}")
print(f"Merges Dict: {final_merges}")